# Model Evaluation & Promotion Gate — WAF Injection Classifier

This notebook is the evaluation gate for the staged transformer checkpoints produced from `edited.ipynb`.

## Scope

- Load the latest staged checkpoints for MiniLM, DistilBERT, and BERT-base
- Validate dataset integrity before inference
- Run deterministic inference on validation and test splits
- Fit temperature scaling on validation logits only
- Report classification, calibration, and WAF operations metrics
- Benchmark latency and throughput on CPU and GPU
- Export versioned artifacts and refresh the standalone predictor module

## Locked Constraints

- Dataset: `v3_907k_cleaned`
- Validation split is used for calibration only
- Test split is used for final evaluation only
- Label order is fixed from training: `['Code Injection', 'Normal', 'Other Attacks', 'SQL Injection']`
- Confidence tiers: `LOW < 0.50`, `MEDIUM 0.50-0.80`, `HIGH > 0.80`
- Checkpoints must be loaded as `state_dict` with `weights_only=True`
- Windows DataLoader workers remain `0`

## Promotion Gates

A model is promotion-ready only if it satisfies all of the following:

- Accuracy >= 0.95
- Macro-F1 >= 0.85
- Benign false-positive rate <= 0.03
- CPU mean end-to-end latency < 100 ms
- Calibration does not degrade materially after temperature scaling

In [1]:
import sys
import torch
import transformers
import sklearn
import numpy
import pandas

print(f"Python       : {sys.version}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"Scikit-learn : {sklearn.__version__}")
print(f"NumPy        : {numpy.__version__}")
print(f"Pandas       : {pandas.__version__}")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA         : {torch.version.cuda}")
else:
    print("GPU          : not available (CPU-only mode)")

version_tuple = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:3])
assert version_tuple >= (2, 6, 0), (
    f"PyTorch >= 2.6.0 required for safe checkpoint loading. Current: {torch.__version__}"
)
print("\n[PASS] Environment and security gate passed.")

Python       : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PyTorch      : 2.6.0+cu124
Transformers : 5.3.0
Scikit-learn : 1.8.0
NumPy        : 2.4.3
Pandas       : 3.0.1
GPU          : NVIDIA GeForce RTX 3060 Laptop GPU
VRAM         : 6.4 GB
CUDA         : 12.4

[PASS] Environment and security gate passed.


In [2]:
import json
import math
import hashlib
import random
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.special
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    auc,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)

import torch
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
print(f"Seed   : {SEED}")
print("Imports and deterministic runtime settings loaded.")

Device : cuda
Seed   : 42
Imports and deterministic runtime settings loaded.


In [3]:
DATA_DIR = Path("..") / "data" / "processed" / "v3_907k_cleaned"
MODEL_STAGING = Path("model_registry") / "staging"
TEXT_COL = "combined_payload"
LABEL_COL = "final_label"
LABEL_NAMES = ["Code Injection", "Normal", "Other Attacks", "SQL Injection"]
NUM_CLASSES = len(LABEL_NAMES)
LOW_THRESHOLD = 0.50
HIGH_THRESHOLD = 0.80
DEBUG_MAX_ROWS = None

TRAINING_METADATA_PATH = DATA_DIR / "training_metadata.json"
AUDIT_LOG_PATH = DATA_DIR / "audit_log.json"
CHECKSUMS_PATH = DATA_DIR / "checksums.txt"

with open(TRAINING_METADATA_PATH, "r", encoding="utf-8") as handle:
    TRAINING_METADATA = json.load(handle)
with open(AUDIT_LOG_PATH, "r", encoding="utf-8") as handle:
    AUDIT_LOG = json.load(handle)

EXPECTED_TEST_ROWS = TRAINING_METADATA["test"]["rows"]
EXPECTED_VAL_ROWS = TRAINING_METADATA["validation"]["rows"]
EXPECTED_TEST_SHA = AUDIT_LOG["files"]["test_parquet"]["sha256"]
REQUIRED_COLS = AUDIT_LOG["statistics"]["final_columns"]
PIPELINE_VERSION = AUDIT_LOG["metadata"]["pipeline_version"]
DATASET_VERSION = "v3_907k_cleaned"

EVAL_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
EVAL_DIR = Path("model_registry") / "eval" / EVAL_TS
PLOT_DIR = EVAL_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset version   : {DATASET_VERSION}")
print(f"Pipeline version  : {PIPELINE_VERSION}")
print(f"Expected val rows : {EXPECTED_VAL_ROWS:,}")
print(f"Expected test rows: {EXPECTED_TEST_ROWS:,}")
print(f"Output directory  : {EVAL_DIR}")
if DEBUG_MAX_ROWS is not None:
    print(f"[DEBUG] Limiting evaluation to {DEBUG_MAX_ROWS:,} rows after integrity checks.")

Dataset version   : v3_907k_cleaned
Pipeline version  : 3.1.0
Expected val rows : 19,661
Expected test rows: 19,505
Output directory  : model_registry\eval\20260312_172840


In [4]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


df_test = pd.read_parquet(DATA_DIR / "test.parquet")
df_val = pd.read_parquet(DATA_DIR / "validation.parquet")

assert len(df_test) == EXPECTED_TEST_ROWS, (
    f"Unexpected test rows: {len(df_test):,} vs {EXPECTED_TEST_ROWS:,}"
)
assert len(df_val) == EXPECTED_VAL_ROWS, (
    f"Unexpected validation rows: {len(df_val):,} vs {EXPECTED_VAL_ROWS:,}"
)

actual_test_sha = sha256_file(DATA_DIR / "test.parquet")
assert actual_test_sha == EXPECTED_TEST_SHA, (
    f"test.parquet SHA256 mismatch. Expected {EXPECTED_TEST_SHA}, got {actual_test_sha}"
)

for split_name, frame in [("validation", df_val), ("test", df_test)]:
    for column in REQUIRED_COLS:
        assert column in frame.columns, f"Missing {column} in {split_name} split"
    observed = set(frame[LABEL_COL].unique())
    assert observed == set(LABEL_NAMES), (
        f"Unexpected labels in {split_name}: {observed}"
    )

label_encoder = LabelEncoder()
label_encoder.classes_ = np.array(LABEL_NAMES)
df_test["label_id"] = label_encoder.transform(df_test[LABEL_COL])
df_val["label_id"] = label_encoder.transform(df_val[LABEL_COL])

if DEBUG_MAX_ROWS is not None:
    df_test = df_test.sample(min(DEBUG_MAX_ROWS, len(df_test)), random_state=SEED).sort_index()
    df_val = df_val.sample(min(DEBUG_MAX_ROWS, len(df_val)), random_state=SEED).sort_index()

print(f"Validation : {len(df_val):,} rows")
print(f"Test       : {len(df_test):,} rows")
print("\nTest label distribution:")
print(df_test[LABEL_COL].value_counts().reindex(LABEL_NAMES).to_string())
print("\n[PASS] Dataset integrity checks passed.")

Validation : 19,661 rows
Test       : 19,505 rows

Test label distribution:
final_label
Code Injection     837
Normal            3658
Other Attacks     6035
SQL Injection     8975

[PASS] Dataset integrity checks passed.


In [5]:
MAX_SEQ_LEN = 128
MODEL_IDS = {
    "minilm": "nreimers/MiniLM-L6-H384-uncased",
    "distilbert": "distilbert-base-uncased",
    "bert-base": "bert-base-uncased",
}
DEFAULT_EVAL_BATCH_SIZES = {
    "minilm": 128,
    "distilbert": 128,
    "bert-base": 64,
}


def parse_run_timestamp(run_dir: Path) -> datetime:
    return datetime.strptime(run_dir.name.rsplit("_", 2)[-2] + run_dir.name.rsplit("_", 1)[-1], "%Y%m%d%H%M%S")


def discover_latest_run(staging_dir: Path, model_key: str) -> Path:
    candidates = [
        run_dir for run_dir in staging_dir.iterdir()
        if run_dir.is_dir() and run_dir.name.startswith(model_key + "_")
    ]
    assert candidates, f"No staged runs found for {model_key} in {staging_dir}"
    candidates.sort(key=parse_run_timestamp, reverse=True)
    return candidates[0]


MODEL_REGISTRY = {}
for model_key, model_id in MODEL_IDS.items():
    run_dir = discover_latest_run(MODEL_STAGING, model_key)
    ckpt_path = run_dir / f"best_{model_key}_ckpt.pt"
    cfg_path = run_dir / "config_used.json"
    git_hash_path = run_dir / "git_hash.txt"
    assert ckpt_path.exists(), f"Checkpoint missing: {ckpt_path}"
    assert cfg_path.exists(), f"Missing config_used.json for {model_key}"
    assert git_hash_path.exists(), f"Missing git_hash.txt for {model_key}"

    with cfg_path.open("r", encoding="utf-8") as handle:
        config_used = json.load(handle)
    git_hash = git_hash_path.read_text(encoding="utf-8").strip()

    MODEL_REGISTRY[model_key] = {
        "model_id": model_id,
        "run_dir": run_dir,
        "ckpt_path": ckpt_path,
        "config_used": config_used,
        "git_hash": git_hash,
        "eval_bs": DEFAULT_EVAL_BATCH_SIZES[model_key],
    }

print("Latest staged checkpoints:")
for key, info in MODEL_REGISTRY.items():
    print(
        f"  {key:12s} run={info['run_dir'].name}  ckpt={info['ckpt_path'].name}  "
        f"train_bs={info['config_used']['per_device_train_batch_size']}  "
        f"seq={info['config_used']['max_seq_len']}  git={info['git_hash'][:8]}"
    )

Latest staged checkpoints:
  minilm       run=minilm_v3_907k_cleaned_20260312_124833  ckpt=best_minilm_ckpt.pt  train_bs=128  seq=128  git=54d248d4
  distilbert   run=distilbert_v3_907k_cleaned_20260312_133755  ckpt=best_distilbert_ckpt.pt  train_bs=64  seq=128  git=54d248d4
  bert-base    run=bert-base_v3_907k_cleaned_20260312_145113  ckpt=best_bert-base_ckpt.pt  train_bs=32  seq=128  git=54d248d4


In [6]:
def preprocess_split(frame: pd.DataFrame, tokenizer, max_len: int) -> dict[str, list[list[int]]]:
    encoded = tokenizer(
        list(frame[TEXT_COL]),
        truncation=True,
        max_length=max_len,
        padding=False,
    )
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
    }


class WAFDataset(Dataset):
    def __init__(self, precomputed: dict[str, list[list[int]]], labels: pd.Series):
        self.input_ids = precomputed["input_ids"]
        self.attention_mask = precomputed["attention_mask"]
        self.labels = labels.reset_index(drop=True)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "input_ids": torch.tensor(self.input_ids[idx], dtype=torch.long),
            "attention_mask": torch.tensor(self.attention_mask[idx], dtype=torch.long),
            "labels": torch.tensor(int(self.labels.iloc[idx]), dtype=torch.long),
        }


def build_eval_dataloader(frame: pd.DataFrame, precomputed: dict[str, list[list[int]]], tokenizer, batch_size: int):
    collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True)
    dataset = WAFDataset(precomputed, frame["label_id"])
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


tokenized_cache = {}
print("Pre-tokenizing validation and test splits per model...")
for key, info in MODEL_REGISTRY.items():
    tokenizer = AutoTokenizer.from_pretrained(info["model_id"])
    MODEL_REGISTRY[key]["tokenizer"] = tokenizer
    t0 = time.time()
    tokenized_cache[key] = {
        "validation": preprocess_split(df_val, tokenizer, MAX_SEQ_LEN),
        "test": preprocess_split(df_test, tokenizer, MAX_SEQ_LEN),
    }
    print(f"  {key:12s} done in {time.time() - t0:.1f}s")

print("Tokenization cache ready.")

Pre-tokenizing validation and test splits per model...
  minilm       done in 1.4s
  distilbert   done in 1.4s
  bert-base    done in 1.2s
Tokenization cache ready.


In [7]:
def load_model_safely(model_key: str, model_id: str, ckpt_path: Path, device: torch.device):
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=NUM_CLASSES)
    state = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    assert isinstance(state, dict), f"Expected state_dict for {model_key}, got {type(state)}"
    assert all(torch.is_tensor(value) for value in state.values()), (
        f"Checkpoint for {model_key} is not a plain tensor state_dict"
    )
    model.load_state_dict(state, strict=True)
    model.to(device)
    model.eval()
    n_params = sum(parameter.numel() for parameter in model.parameters())
    print(f"  Loaded {model_key}: {n_params:,} parameters")
    return model


@torch.no_grad()
def collect_inference(model, dataloader, device: torch.device):
    logits_chunks = []
    labels_chunks = []
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"]
        with autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device.type == "cuda")):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits_chunks.append(outputs.logits.float().cpu())
        labels_chunks.append(labels)

    logits = torch.cat(logits_chunks, dim=0).numpy()
    labels = torch.cat(labels_chunks, dim=0).numpy()
    probs = scipy.special.softmax(logits, axis=1)
    preds = np.argmax(probs, axis=1)
    max_probs = probs.max(axis=1)
    return {
        "logits": logits,
        "probs": probs,
        "preds": preds,
        "labels": labels,
        "max_probs": max_probs,
    }


results = {}
for key, info in MODEL_REGISTRY.items():
    print(f"\n{'=' * 72}\nInference: {key}\n{'=' * 72}")
    model = load_model_safely(key, info["model_id"], info["ckpt_path"], DEVICE)
    tokenizer = info["tokenizer"]
    batch_size = info["eval_bs"]

    val_loader = build_eval_dataloader(df_val, tokenized_cache[key]["validation"], tokenizer, batch_size)
    test_loader = build_eval_dataloader(df_test, tokenized_cache[key]["test"], tokenizer, batch_size)

    t0 = time.time()
    val_outputs = collect_inference(model, val_loader, DEVICE)
    print(f"  Validation inference: {time.time() - t0:.1f}s for {len(val_outputs['labels']):,} rows")

    t0 = time.time()
    test_outputs = collect_inference(model, test_loader, DEVICE)
    print(f"  Test inference      : {time.time() - t0:.1f}s for {len(test_outputs['labels']):,} rows")

    results[key] = {
        "validation": val_outputs,
        "test": test_outputs,
    }

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nInference finished for {len(results)} models.")


Inference: minilm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nreimers/MiniLM-L6-H384-uncased
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded minilm: 22,714,756 parameters
  Validation inference: 7.1s for 19,661 rows
  Test inference      : 6.5s for 19,505 rows

Inference: distilbert


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded distilbert: 66,956,548 parameters
  Validation inference: 14.5s for 19,661 rows
  Test inference      : 14.2s for 19,505 rows

Inference: bert-base


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded bert-base: 109,485,316 parameters
  Validation inference: 28.1s for 19,661 rows
  Test inference      : 27.2s for 19,505 rows

Inference finished for 3 models.


In [8]:
def compute_ece_and_mce(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15):
    preds = np.argmax(probs, axis=1)
    confidences = probs.max(axis=1)
    correctness = (preds == labels).astype(float)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    mce = 0.0
    bin_rows = []

    for idx in range(n_bins):
        lower = bin_edges[idx]
        upper = bin_edges[idx + 1]
        mask = (confidences > lower) & (confidences <= upper)
        if idx == 0:
            mask = (confidences >= lower) & (confidences <= upper)
        if not mask.any():
            bin_rows.append({"bin": idx, "count": 0, "avg_conf": 0.0, "avg_acc": 0.0, "gap": 0.0})
            continue
        avg_conf = float(confidences[mask].mean())
        avg_acc = float(correctness[mask].mean())
        gap = abs(avg_conf - avg_acc)
        weight = float(mask.mean())
        ece += weight * gap
        mce = max(mce, gap)
        bin_rows.append({
            "bin": idx,
            "count": int(mask.sum()),
            "avg_conf": avg_conf,
            "avg_acc": avg_acc,
            "gap": gap,
        })

    return float(ece), float(mce), bin_rows


def fit_temperature(val_logits: np.ndarray, val_labels: np.ndarray):
    logits_tensor = torch.tensor(val_logits, dtype=torch.float32)
    labels_tensor = torch.tensor(val_labels, dtype=torch.long)
    log_temperature = torch.nn.Parameter(torch.zeros(1, dtype=torch.float32))
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=100, line_search_fn="strong_wolfe")

    def closure():
        optimizer.zero_grad()
        temperature = torch.exp(log_temperature).clamp(1e-3, 100.0)
        loss = criterion(logits_tensor / temperature, labels_tensor)
        if not torch.isfinite(loss):
            raise RuntimeError("Temperature fitting produced a non-finite loss")
        loss.backward()
        return loss

    before_nll = float(criterion(logits_tensor, labels_tensor).item())
    before_probs = scipy.special.softmax(val_logits, axis=1)
    before_ece, before_mce, _ = compute_ece_and_mce(before_probs, val_labels)
    optimizer.step(closure)

    temperature_value = float(torch.exp(log_temperature.detach()).item())
    assert math.isfinite(temperature_value) and temperature_value > 0.0, (
        f"Invalid fitted temperature: {temperature_value}"
    )

    after_probs = scipy.special.softmax(val_logits / temperature_value, axis=1)
    after_nll = float(criterion(logits_tensor / temperature_value, labels_tensor).item())
    after_ece, after_mce, bin_rows = compute_ece_and_mce(after_probs, val_labels)

    return {
        "temperature": temperature_value,
        "before_nll": before_nll,
        "after_nll": after_nll,
        "before_ece": before_ece,
        "after_ece": after_ece,
        "before_mce": before_mce,
        "after_mce": after_mce,
        "bin_rows": bin_rows,
    }


for key, payload in results.items():
    calibration = fit_temperature(payload["validation"]["logits"], payload["validation"]["labels"])
    payload["calibration"] = calibration
    payload["validation"]["probs_calibrated"] = scipy.special.softmax(
        payload["validation"]["logits"] / calibration["temperature"], axis=1
    )
    payload["test"]["probs_calibrated"] = scipy.special.softmax(
        payload["test"]["logits"] / calibration["temperature"], axis=1
    )
    payload["test"]["preds_calibrated"] = np.argmax(payload["test"]["probs_calibrated"], axis=1)
    payload["test"]["max_probs_calibrated"] = payload["test"]["probs_calibrated"].max(axis=1)
    print(
        f"{key:12s} T={calibration['temperature']:.4f}  "
        f"val_NLL {calibration['before_nll']:.4f}->{calibration['after_nll']:.4f}  "
        f"val_ECE {calibration['before_ece']:.4f}->{calibration['after_ece']:.4f}"
    )

print("\nTemperature scaling complete.")

minilm       T=0.5559  val_NLL 0.0615->0.0287  val_ECE 0.0385->0.0030
distilbert   T=0.5969  val_NLL 0.0341->0.0189  val_ECE 0.0180->0.0020
bert-base    T=0.5967  val_NLL 0.0347->0.0189  val_ECE 0.0185->0.0017

Temperature scaling complete.


In [9]:
def confidence_tier(max_prob: float) -> str:
    if max_prob < LOW_THRESHOLD:
        return "LOW"
    if max_prob <= HIGH_THRESHOLD:
        return "MEDIUM"
    return "HIGH"


def multiclass_brier_score(probs: np.ndarray, labels: np.ndarray) -> float:
    one_hot = np.eye(NUM_CLASSES)[labels]
    return float(np.mean(np.sum((probs - one_hot) ** 2, axis=1)))


def compute_operational_metrics(labels: np.ndarray, preds: np.ndarray, max_probs: np.ndarray):
    normal_idx = LABEL_NAMES.index("Normal")
    normal_mask = labels == normal_idx
    attack_mask = labels != normal_idx
    benign_fp_rate = float((preds[normal_mask] != normal_idx).mean()) if normal_mask.any() else 0.0
    benign_high_conf_rate = float(((preds[normal_mask] != normal_idx) & (max_probs[normal_mask] > HIGH_THRESHOLD)).mean()) if normal_mask.any() else 0.0
    attack_detection_rate = float((preds[attack_mask] != normal_idx).mean()) if attack_mask.any() else 0.0

    tier_rows = {}
    for tier_name in ["LOW", "MEDIUM", "HIGH"]:
        tier_mask = np.array([confidence_tier(value) == tier_name for value in max_probs])
        count = int(tier_mask.sum())
        tier_rows[tier_name] = {
            "count": count,
            "pct": round(count / len(labels) * 100, 4),
            "accuracy": round(float((preds[tier_mask] == labels[tier_mask]).mean()), 6) if count else 0.0,
            "predicted_class_distribution": {
                label_name: int((preds[tier_mask] == idx).sum()) for idx, label_name in enumerate(LABEL_NAMES)
            },
        }

    return {
        "benign_false_positive_rate": round(benign_fp_rate, 6),
        "benign_high_confidence_false_block_rate": round(benign_high_conf_rate, 6),
        "attack_detection_rate": round(attack_detection_rate, 6),
        "confidence_tiers": tier_rows,
    }


def compute_metric_bundle(model_key: str, variant: str, labels: np.ndarray, preds: np.ndarray, probs: np.ndarray, temperature: float):
    max_probs = probs.max(axis=1)
    ece, mce, bin_rows = compute_ece_and_mce(probs, labels)
    precision, recall, f1_values, support = precision_recall_fscore_support(
        labels,
        preds,
        labels=list(range(NUM_CLASSES)),
        zero_division=0,
    )

    per_class = {}
    for idx, label_name in enumerate(LABEL_NAMES):
        per_class[label_name] = {
            "precision": round(float(precision[idx]), 6),
            "recall": round(float(recall[idx]), 6),
            "f1": round(float(f1_values[idx]), 6),
            "support": int(support[idx]),
            "pr_auc": round(float(average_precision_score((labels == idx).astype(int), probs[:, idx])), 6),
        }

    bundle = {
        "model": model_key,
        "variant": variant,
        "accuracy": round(float(accuracy_score(labels, preds)), 6),
        "macro_f1": round(float(f1_score(labels, preds, average="macro")), 6),
        "weighted_f1": round(float(f1_score(labels, preds, average="weighted")), 6),
        "mcc": round(float(matthews_corrcoef(labels, preds)), 6),
        "cohen_kappa": round(float(cohen_kappa_score(labels, preds)), 6),
        "log_loss": round(float(log_loss(labels, probs, labels=list(range(NUM_CLASSES)))), 6),
        "roc_auc_ovr": round(float(roc_auc_score(labels, probs, multi_class="ovr", average="macro")), 6),
        "brier_score": round(multiclass_brier_score(probs, labels), 6),
        "ece": round(float(ece), 6),
        "mce": round(float(mce), 6),
        "temperature": round(float(temperature), 6),
        "per_class": per_class,
        "calibration_bins": bin_rows,
        "operational": compute_operational_metrics(labels, preds, max_probs),
    }
    return bundle


all_metrics = {}
for key, payload in results.items():
    test_labels = payload["test"]["labels"]

    uncalibrated = compute_metric_bundle(
        key,
        "uncalibrated",
        test_labels,
        payload["test"]["preds"],
        payload["test"]["probs"],
        1.0,
    )
    calibrated = compute_metric_bundle(
        key,
        "calibrated",
        test_labels,
        payload["test"]["preds_calibrated"],
        payload["test"]["probs_calibrated"],
        payload["calibration"]["temperature"],
    )

    all_metrics[f"{key}_uncalibrated"] = uncalibrated
    all_metrics[f"{key}_calibrated"] = calibrated

    print(
        f"{key:12s} acc={calibrated['accuracy']:.4f}  macro_f1={calibrated['macro_f1']:.4f}  "
        f"ECE={calibrated['ece']:.4f}  benign_FPR={calibrated['operational']['benign_false_positive_rate']:.4f}"
    )

print(f"\nComputed metrics for {len(all_metrics)} model/variant combinations.")

minilm       acc=0.9903  macro_f1=0.9867  ECE=0.0051  benign_FPR=0.0016
distilbert   acc=0.9926  macro_f1=0.9885  ECE=0.0033  benign_FPR=0.0019
bert-base    acc=0.9923  macro_f1=0.9877  ECE=0.0036  benign_FPR=0.0016

Computed metrics for 6 model/variant combinations.


In [10]:
sns.set_theme(style="whitegrid", font_scale=1.05)

for key, payload in results.items():
    labels = payload["test"]["labels"]
    probs = payload["test"]["probs_calibrated"]
    preds = payload["test"]["preds_calibrated"]

    cm = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    cm_normalized = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), a_min=1, a_max=None)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[0])
    axes[0].set_title(f"{key} confusion matrix")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("True")

    sns.heatmap(cm_normalized, annot=True, fmt=".2%", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=axes[1])
    axes[1].set_title(f"{key} normalized confusion matrix")
    axes[1].set_xlabel("Predicted")
    axes[1].set_ylabel("True")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"confmat_{key}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 7))
    for idx, label_name in enumerate(LABEL_NAMES):
        fpr, tpr, _ = roc_curve((labels == idx).astype(int), probs[:, idx])
        ax.plot(fpr, tpr, label=f"{label_name} (AUC={auc(fpr, tpr):.4f})")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.35)
    ax.set_title(f"ROC curves - {key}")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"roc_{key}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 7))
    for idx, label_name in enumerate(LABEL_NAMES):
        precision_curve, recall_curve, _ = precision_recall_curve((labels == idx).astype(int), probs[:, idx])
        ap_score = average_precision_score((labels == idx).astype(int), probs[:, idx])
        ax.plot(recall_curve, precision_curve, label=f"{label_name} (AP={ap_score:.4f})")
    ax.set_title(f"Precision-recall curves - {key}")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(loc="lower left")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"pr_{key}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

print(f"Saved confusion, ROC, and PR plots to {PLOT_DIR}")

Saved confusion, ROC, and PR plots to model_registry\eval\20260312_172840\plots


In [11]:
for key, payload in results.items():
    labels = payload["test"]["labels"]

    fig, ax = plt.subplots(figsize=(7, 7))
    for variant_name, probs, linestyle in [
        ("uncalibrated", payload["test"]["probs"], "--"),
        ("calibrated", payload["test"]["probs_calibrated"], "-"),
    ]:
        ece, _, bin_rows = compute_ece_and_mce(probs, labels)
        xs = [row["avg_conf"] for row in bin_rows if row["count"] > 0]
        ys = [row["avg_acc"] for row in bin_rows if row["count"] > 0]
        ax.plot(xs, ys, linestyle, marker="o", label=f"{variant_name} (ECE={ece:.4f})")

    ax.plot([0, 1], [0, 1], "k--", alpha=0.35, label="Perfect calibration")
    ax.set_title(f"Reliability diagram - {key}")
    ax.set_xlabel("Mean predicted confidence")
    ax.set_ylabel("Empirical accuracy")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"reliability_{key}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    max_probs = payload["test"]["max_probs_calibrated"]
    correct = (payload["test"]["preds_calibrated"] == labels).astype(float)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(max_probs, bins=50, color="steelblue", alpha=0.8, edgecolor="white")
    ax.axvline(LOW_THRESHOLD, color="firebrick", linestyle="--", linewidth=2, label="LOW/MEDIUM = 0.50")
    ax.axvline(HIGH_THRESHOLD, color="darkgreen", linestyle="--", linewidth=2, label="MEDIUM/HIGH = 0.80")

    for tier_name, left, right, color in [
        ("LOW", 0.00, 0.50, "#ffd9d9"),
        ("MEDIUM", 0.50, 0.80, "#fff4c2"),
        ("HIGH", 0.80, 1.00, "#d7f7d7"),
    ]:
        tier_mask = np.array([confidence_tier(value) == tier_name for value in max_probs])
        count = int(tier_mask.sum())
        accuracy = float(correct[tier_mask].mean()) if count else 0.0
        ax.axvspan(left, right, color=color, alpha=0.15)
        ax.text(
            (left + right) / 2,
            ax.get_ylim()[1] * 0.92,
            f"{tier_name}\nn={count}\nacc={accuracy:.3f}",
            ha="center",
            va="top",
            fontsize=9,
            bbox={"boxstyle": "round,pad=0.25", "facecolor": color, "alpha": 0.8},
        )

    ax.set_title(f"Confidence distribution - {key}")
    ax.set_xlabel("Max calibrated probability")
    ax.set_ylabel("Count")
    ax.legend(loc="upper left")
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f"confidence_hist_{key}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

print(f"Saved calibration and confidence plots to {PLOT_DIR}")

Saved calibration and confidence plots to model_registry\eval\20260312_172840\plots


In [12]:
def percentile_summary(values_ms: list[float]) -> dict[str, float]:
    arr = np.array(values_ms, dtype=np.float64)
    return {
        "mean_ms": round(float(arr.mean()), 4),
        "p50_ms": round(float(np.percentile(arr, 50)), 4),
        "p95_ms": round(float(np.percentile(arr, 95)), 4),
        "p99_ms": round(float(np.percentile(arr, 99)), 4),
        "std_ms": round(float(arr.std()), 4),
    }


def benchmark_end_to_end(model, tokenizer, sample_text: str, device: torch.device, n: int = 200, warmup: int = 30):
    model = model.to(device).eval()
    timings = []
    for _ in range(warmup):
        encoded = tokenizer(sample_text, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt", padding=True)
        encoded = {name: tensor.to(device) for name, tensor in encoded.items()}
        with torch.no_grad():
            model(**encoded)
        if device.type == "cuda":
            torch.cuda.synchronize()

    for _ in range(n):
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        encoded = tokenizer(sample_text, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt", padding=True)
        encoded = {name: tensor.to(device) for name, tensor in encoded.items()}
        with torch.no_grad():
            model(**encoded)
        if device.type == "cuda":
            torch.cuda.synchronize()
        timings.append((time.perf_counter() - start) * 1000.0)
    return percentile_summary(timings)


def benchmark_model_only(model, tokenizer, sample_text: str, device: torch.device, n: int = 200, warmup: int = 30):
    model = model.to(device).eval()
    encoded = tokenizer(sample_text, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt", padding=True)
    encoded = {name: tensor.to(device) for name, tensor in encoded.items()}
    timings = []

    for _ in range(warmup):
        with torch.no_grad():
            model(**encoded)
        if device.type == "cuda":
            torch.cuda.synchronize()

    for _ in range(n):
        if device.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            model(**encoded)
        if device.type == "cuda":
            torch.cuda.synchronize()
        timings.append((time.perf_counter() - start) * 1000.0)
    return percentile_summary(timings)


def benchmark_throughput(model, tokenizer, sample_text: str, device: torch.device, batch_sizes=(1, 8, 16, 32, 64), repeats: int = 30):
    model = model.to(device).eval()
    throughput = {}
    for batch_size in batch_sizes:
        batch = [sample_text] * batch_size
        encoded = tokenizer(batch, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt", padding=True)
        encoded = {name: tensor.to(device) for name, tensor in encoded.items()}
        for _ in range(5):
            with torch.no_grad():
                model(**encoded)
            if device.type == "cuda":
                torch.cuda.synchronize()
        timings = []
        for _ in range(repeats):
            if device.type == "cuda":
                torch.cuda.synchronize()
            start = time.perf_counter()
            with torch.no_grad():
                model(**encoded)
            if device.type == "cuda":
                torch.cuda.synchronize()
            timings.append(time.perf_counter() - start)
        mean_seconds = float(np.mean(timings))
        throughput[str(batch_size)] = round(batch_size / mean_seconds, 4)
    return throughput


sample_idx = int((df_test[TEXT_COL].str.len() - df_test[TEXT_COL].str.len().median()).abs().idxmin())
sample_text = df_test.loc[sample_idx, TEXT_COL]
print(f"Latency benchmark sample length: {len(sample_text)} characters")

latency_results = {}
for key, info in MODEL_REGISTRY.items():
    print(f"\nBenchmarking {key}...")
    model = load_model_safely(key, info["model_id"], info["ckpt_path"], DEVICE)
    tokenizer = info["tokenizer"]

    latency_results[key] = {
        "cpu_end_to_end": benchmark_end_to_end(model, tokenizer, sample_text, torch.device("cpu"), n=150, warmup=25),
        "cpu_model_only": benchmark_model_only(model, tokenizer, sample_text, torch.device("cpu"), n=150, warmup=25),
        "cpu_throughput": benchmark_throughput(model, tokenizer, sample_text, torch.device("cpu")),
    }

    if torch.cuda.is_available():
        latency_results[key]["gpu_end_to_end"] = benchmark_end_to_end(model, tokenizer, sample_text, DEVICE, n=300, warmup=40)
        latency_results[key]["gpu_model_only"] = benchmark_model_only(model, tokenizer, sample_text, DEVICE, n=300, warmup=40)
        latency_results[key]["gpu_throughput"] = benchmark_throughput(model, tokenizer, sample_text, DEVICE)

    cpu_mean = latency_results[key]["cpu_end_to_end"]["mean_ms"]
    status = "PASS" if cpu_mean < 100.0 else "FAIL"
    print(f"  CPU end-to-end mean={cpu_mean:.2f} ms [{status}]")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nLatency and throughput benchmarking complete.")

Latency benchmark sample length: 87 characters

Benchmarking minilm...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: nreimers/MiniLM-L6-H384-uncased
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded minilm: 22,714,756 parameters
  CPU end-to-end mean=8.11 ms [PASS]

Benchmarking distilbert...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded distilbert: 66,956,548 parameters
  CPU end-to-end mean=21.45 ms [PASS]

Benchmarking bert-base...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded bert-base: 109,485,316 parameters
  CPU end-to-end mean=41.52 ms [PASS]

Latency and throughput benchmarking complete.


In [13]:
comparison_rows = []
promotion_summary = {}

for model_key in MODEL_REGISTRY:
    calibrated_metrics = all_metrics[f"{model_key}_calibrated"]
    calibration = results[model_key]["calibration"]
    cpu_latency = latency_results[model_key]["cpu_end_to_end"]

    passes = {
        "accuracy": calibrated_metrics["accuracy"] >= 0.95,
        "macro_f1": calibrated_metrics["macro_f1"] >= 0.85,
        "benign_false_positive_rate": calibrated_metrics["operational"]["benign_false_positive_rate"] <= 0.03,
        "cpu_mean_latency": cpu_latency["mean_ms"] < 100.0,
        "calibration_non_regression": calibration["after_ece"] <= calibration["before_ece"] + 0.005,
    }

    promotion_summary[model_key] = {
        "ready_for_promotion": all(passes.values()),
        "gates": passes,
        "temperature": round(calibration["temperature"], 6),
        "git_hash": MODEL_REGISTRY[model_key]["git_hash"],
        "run_dir": MODEL_REGISTRY[model_key]["run_dir"].name,
    }

    comparison_rows.append({
        "model": model_key,
        "run_dir": MODEL_REGISTRY[model_key]["run_dir"].name,
        "git_hash": MODEL_REGISTRY[model_key]["git_hash"],
        "accuracy": calibrated_metrics["accuracy"],
        "macro_f1": calibrated_metrics["macro_f1"],
        "weighted_f1": calibrated_metrics["weighted_f1"],
        "mcc": calibrated_metrics["mcc"],
        "ece": calibrated_metrics["ece"],
        "benign_false_positive_rate": calibrated_metrics["operational"]["benign_false_positive_rate"],
        "benign_high_confidence_false_block_rate": calibrated_metrics["operational"]["benign_high_confidence_false_block_rate"],
        "attack_detection_rate": calibrated_metrics["operational"]["attack_detection_rate"],
        "cpu_end_to_end_mean_ms": cpu_latency["mean_ms"],
        "cpu_model_only_mean_ms": latency_results[model_key]["cpu_model_only"]["mean_ms"],
        "temperature": round(calibration["temperature"], 6),
        "promotion_ready": all(passes.values()),
    })

    for variant in ["uncalibrated", "calibrated"]:
        with (EVAL_DIR / f"eval_results_{model_key}_{variant}.json").open("w", encoding="utf-8") as handle:
            json.dump(all_metrics[f"{model_key}_{variant}"], handle, indent=2)

    test_payload = results[model_key]["test"]
    calibrated_probs = test_payload["probs_calibrated"]
    calibrated_preds = test_payload["preds_calibrated"]
    raw_output = pd.DataFrame({
        "payload_hash": df_test["payload_hash"].values,
        "true_label_id": test_payload["labels"],
        "true_label_name": [LABEL_NAMES[idx] for idx in test_payload["labels"]],
        "pred_label_id": calibrated_preds,
        "pred_label_name": [LABEL_NAMES[idx] for idx in calibrated_preds],
        "max_prob": calibrated_probs.max(axis=1),
        "confidence_tier": [confidence_tier(value) for value in calibrated_probs.max(axis=1)],
        "model": model_key,
        "calibrated": True,
    })
    for idx, label_name in enumerate(LABEL_NAMES):
        safe_label = label_name.lower().replace(" ", "_")
        raw_output[f"logit_{safe_label}"] = test_payload["logits"][:, idx]
        raw_output[f"prob_{safe_label}"] = calibrated_probs[:, idx]
    raw_output.to_parquet(EVAL_DIR / f"eval_raw_outputs_{model_key}.parquet", index=False)

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    by=["promotion_ready", "macro_f1", "benign_false_positive_rate", "cpu_end_to_end_mean_ms"],
    ascending=[False, False, True, True],
)
comparison_df.to_csv(EVAL_DIR / "model_comparison.csv", index=False)

best_overall_model = comparison_df.iloc[0]["model"]
fastest_model = comparison_df.sort_values("cpu_end_to_end_mean_ms", ascending=True).iloc[0]["model"]
highest_accuracy_model = comparison_df.sort_values("accuracy", ascending=False).iloc[0]["model"]

summary_payload = {
    "timestamp": EVAL_TS,
    "dataset_version": DATASET_VERSION,
    "pipeline_version": PIPELINE_VERSION,
    "best_overall_model": best_overall_model,
    "highest_accuracy_model": highest_accuracy_model,
    "lowest_latency_model": fastest_model,
    "promotion_summary": promotion_summary,
}
with (EVAL_DIR / "promotion_summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary_payload, handle, indent=2)
with (EVAL_DIR / "latency_results.json").open("w", encoding="utf-8") as handle:
    json.dump(latency_results, handle, indent=2)
with (EVAL_DIR / "environment.json").open("w", encoding="utf-8") as handle:
    json.dump({
        "python": sys.version,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "sklearn": sklearn.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "device": str(DEVICE),
        "cuda": torch.version.cuda,
        "seed": SEED,
    }, handle, indent=2)

print("Model comparison:")
print(comparison_df.to_string(index=False))
print(f"\nBest overall model : {best_overall_model}")
print(f"Highest accuracy   : {highest_accuracy_model}")
print(f"Lowest latency     : {fastest_model}")
print(f"Artifacts saved to : {EVAL_DIR}")

Model comparison:
     model                                    run_dir                                 git_hash  accuracy  macro_f1  weighted_f1      mcc      ece  benign_false_positive_rate  benign_high_confidence_false_block_rate  attack_detection_rate  cpu_end_to_end_mean_ms  cpu_model_only_mean_ms  temperature  promotion_ready
distilbert distilbert_v3_907k_cleaned_20260312_133755 54d248d4edcfda2b30a8d96b5800a2bfb570d456  0.992566  0.988523     0.992591 0.988707 0.003296                    0.001914                                 0.001093               0.997665                 21.4489                 21.0193     0.596868             True
 bert-base  bert-base_v3_907k_cleaned_20260312_145113 54d248d4edcfda2b30a8d96b5800a2bfb570d456  0.992310  0.987690     0.992339 0.988319 0.003648                    0.001640                                 0.001093               0.997476                 41.5233                 41.3906     0.596741             True
    minilm     minilm_v3_907k_clea

In [14]:
PREDICTOR_MODULE = '''"""
predict_attack - Production inference wrapper for the staged WAF classifier.
Generated from ml_model/evaluate.ipynb.
"""

import json
import time
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

LABEL_NAMES = ["Code Injection", "Normal", "Other Attacks", "SQL Injection"]
NUM_CLASSES = len(LABEL_NAMES)
MAX_SEQ_LEN = 128
LOW_THRESHOLD = 0.50
HIGH_THRESHOLD = 0.80
MODEL_IDS = {
    "minilm": "nreimers/MiniLM-L6-H384-uncased",
    "distilbert": "distilbert-base-uncased",
    "bert-base": "bert-base-uncased",
}


def _discover_latest_run(staging_dir: Path, model_key: str) -> Path:
    candidates = [
        run_dir for run_dir in staging_dir.iterdir()
        if run_dir.is_dir() and run_dir.name.startswith(model_key + "_")
    ]
    if not candidates:
        raise FileNotFoundError(f"No staged run found for {model_key} in {staging_dir}")
    candidates.sort(key=lambda path: path.name, reverse=True)
    return candidates[0]


def _load_temperature(eval_dir: Path, model_key: str) -> float:
    if not eval_dir.exists():
        return 1.0
    for run_dir in sorted([path for path in eval_dir.iterdir() if path.is_dir()], key=lambda path: path.name, reverse=True):
        result_path = run_dir / f"eval_results_{model_key}_calibrated.json"
        if result_path.exists():
            with result_path.open("r", encoding="utf-8") as handle:
                return float(json.load(handle).get("temperature", 1.0))
    return 1.0


def load_model(model_key: str, staging_dir=None, device="cpu"):
    if model_key not in MODEL_IDS:
        raise KeyError(f"Unknown model key: {model_key}")
    if staging_dir is None:
        staging_dir = Path(__file__).resolve().parent.parent / "model_registry" / "staging"
    staging_dir = Path(staging_dir)
    run_dir = _discover_latest_run(staging_dir, model_key)
    ckpt_path = run_dir / f"best_{model_key}_ckpt.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Missing checkpoint: {ckpt_path}")

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_IDS[model_key], num_labels=NUM_CLASSES)
    state = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_IDS[model_key])
    temperature = _load_temperature(staging_dir.parent / "eval", model_key)
    return model, tokenizer, temperature


def predict_attack(text: str, model, tokenizer, device="cpu", temperature: float = 1.0, return_latency: bool = True):
    if isinstance(device, str):
        device = torch.device(device)

    start = time.perf_counter()
    encoded = tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt", padding=True)
    encoded = {name: tensor.to(device) for name, tensor in encoded.items()}

    with torch.no_grad():
        logits = model(**encoded).logits.float().cpu()

    probs = torch.softmax(logits / float(temperature), dim=-1).squeeze().tolist()
    pred_idx = int(np.argmax(probs))
    max_prob = float(max(probs))
    if max_prob < LOW_THRESHOLD:
        tier = "LOW"
    elif max_prob <= HIGH_THRESHOLD:
        tier = "MEDIUM"
    else:
        tier = "HIGH"

    payload = {
        "label": LABEL_NAMES[pred_idx],
        "label_idx": pred_idx,
        "probs": [round(float(value), 6) for value in probs],
        "max_prob": round(max_prob, 6),
        "tier": tier,
    }
    if return_latency:
        payload["latency_ms"] = round((time.perf_counter() - start) * 1000.0, 3)
    return payload
'''

predictor_path = Path("inference") / "predict_attack.py"
predictor_path.write_text(PREDICTOR_MODULE + "\n", encoding="utf-8")
print(f"Predictor module written to {predictor_path}")

smoke_model_key = "distilbert"
smoke_model = load_model_safely(
    smoke_model_key,
    MODEL_REGISTRY[smoke_model_key]["model_id"],
    MODEL_REGISTRY[smoke_model_key]["ckpt_path"],
    DEVICE,
)
smoke_tokenizer = MODEL_REGISTRY[smoke_model_key]["tokenizer"]
smoke_temperature = results[smoke_model_key]["calibration"]["temperature"]

for sample in [
    "SELECT * FROM users WHERE 1=1 --",
    "GET /index.html HTTP/1.1",
    "<script>alert(1)</script>",
    "../../etc/passwd",
]:
    encoded = smoke_tokenizer(sample, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt", padding=True)
    encoded = {name: tensor.to(DEVICE) for name, tensor in encoded.items()}
    with torch.no_grad():
        logits = smoke_model(**encoded).logits.float().cpu()
    probs = torch.softmax(logits / smoke_temperature, dim=-1).squeeze().tolist()
    label = LABEL_NAMES[int(np.argmax(probs))]
    max_prob = float(max(probs))
    tier = confidence_tier(max_prob)
    print(f"{tier:6s} {max_prob:.4f} {label:20s} {sample[:60]}")

del smoke_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("predict_attack module refreshed and smoke-tested.")

Predictor module written to inference\predict_attack.py


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Loaded distilbert: 66,956,548 parameters
HIGH   0.9973 SQL Injection        SELECT * FROM users WHERE 1=1 --
HIGH   0.9417 Other Attacks        GET /index.html HTTP/1.1
HIGH   0.9998 Code Injection       <script>alert(1)</script>
MEDIUM 0.5795 Normal               ../../etc/passwd
predict_attack module refreshed and smoke-tested.


## Summary and Next Actions

Review the generated artifacts in `model_registry/eval/<timestamp>/`.

Primary outputs:

- `model_comparison.csv` for the side-by-side model decision table
- `promotion_summary.json` for gate-by-gate readiness
- `eval_results_<model>_<variant>.json` for detailed metrics
- `eval_raw_outputs_<model>.parquet` for row-level audit trails
- `plots/*.png` for confusion, ROC, PR, calibration, and confidence visuals
- `inference/predict_attack.py` for application integration

Recommended follow-up after a full run:

1. Promote the best model only if all gates pass.
2. If benign false-positive rate is too high, tune `ci_weight_scale` before any threshold changes.
3. If CPU latency misses target, export the winning model to ONNX Runtime.
4. If calibration improves only marginally, still keep the calibrated output path because enforcement logic depends on threshold semantics.
5. Run multi-seed stability checks for the final candidate before production deployment.